##Read & Write en Delta Lake

In [0]:

%sql
--Creamos la base de datos de Movie_demo
CREATE SCHEMA IF NOT EXISTS movie_demo
MANAGED LOCATION "abfss://demo@lsdata01.dfs.core.windows.net/

"

In [0]:
%run "../includes/librerias"

In [0]:
import pyspark.sql.functions

movie_schema = StructType ( fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )

#Leemos un archivo y cremoa el dataframe movie_df
movie_df = spark.read\
                .option("header", True)\
                .schema(movie_schema) \
                .csv("abfss://bronze@lsdata01.dfs.core.windows.net/2024-12-30/movie.csv")

In [0]:
#1. Escribir datos en Delta Lake (Managed Table)
# Utilizamos el DataFrame movie_df y Creamos una tabla administrada en formato delta
movie_df.write.format("delta").mode("overwrite").saveAsTable("movie_demo.movies_managed")

#2. Escribir datos en Delta Lake (External Table)
# O podemos Guarda los datos del dataframe movie_df en un directorio en formato delta
movie_df.write.format("delta").mode("overwrite").save("abfss://demo@lsdata01.dfs.core.windows.net/movies_external")

In [0]:
%sql
--3. con los datos que se guardaron en la carpeta external, podemos Crear una tabla utilizando los ficheros delta
create table movie_demo.movies_external
using delta
location "abfss://demo@lsdata01.dfs.core.windows.net/movies_external"

####3. Leer datos en Delta Lake (carpeta)

In [0]:
#Cargamos los datos de la carpeta delta en un dataframe
movies_external_df = spark.read.format("delta").load("abfss://demo@lsdata01.dfs.core.windows.net/movies_external")

####3. Leer datos en Delta Lake (Table)

In [0]:
#Creamos una tabla administrada por una particion
movie_df.write.format("delta").mode("overwrite").partitionBy("yearReleaseDate").saveAsTable("movie_demo.movies_partitioned")


In [0]:
%sql
describe extended movie_demo.movies_partitioned

#### Update desde Delta Lake

In [0]:
%sql
SELECT * 
FROM movie_demo.movies_managed 




In [0]:
%sql
-- Actualiza registros utilizando SQL

UPDATE movie_demo.movies_managed
SET durationTime = 60
WHERE yearReleaseDate = 2012


In [0]:
%sql
select durationTime from movie_demo.movies_managed where yearReleaseDate = 2012

In [0]:
##Actualiza registros utilizando SPython
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "movie_demo.movies_managed")
deltaTable.update(
                    condition = "yearReleaseDate = 2013",
                    set = {"durationTime": "100"}
                )

In [0]:
%sql
select durationTime from movie_demo.movies_managed where yearReleaseDate = 2013

In [0]:
%sql
-- Borra registros utilizando SQL

delete from movie_demo.movies_managed
WHERE yearReleaseDate = 2014;

select * from movie_demo.movies_managed
WHERE yearReleaseDate = 2014;


In [0]:
##Borra registros utilizando Python
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "movie_demo.movies_managed")
deltaTable.delete(
                    "yearReleaseDate = 2015"
                 )

In [0]:
%sql
select * from movie_demo.movies_managed
WHERE yearReleaseDate = 2015;

###Merge

In [0]:
import pyspark.sql.functions
from pyspark.sql.functions import upper

movie_schema = StructType ( fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )

#Leemos un archivo y cremoa el dataframe movie_df para el dia 1
movie_day1_df = spark.read\
                    .option("header", True)\
                    .schema(movie_schema) \
                    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/2024-12-30/movie.csv")\
                    .filter("yearReleaseDate < 2000")\
                    .select("movieId", "title", "yearReleaseDate", "releaseDate", "durationTime")

#Leemos un archivo y cremoa el dataframe movie_df para el dia 2
movie_day2_df = spark.read\
                    .option("header", True)\
                    .schema(movie_schema) \
                    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/2024-12-30/movie.csv")\
                    .filter("yearReleaseDate between 1998 and 2005")\
                    .select("movieId", upper("title").alias("title"), "yearReleaseDate", "releaseDate", "durationTime")

#Leemos un archivo y cremoa el dataframe movie_df para el dia 3
movie_day3_df = spark.read\
                    .option("header", True)\
                    .schema(movie_schema) \
                    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/2024-12-30/movie.csv")\
                    .filter("yearReleaseDate between 1983 and 1988 or yearReleaseDate between 2006 and 2010") \
                    .select("movieId", upper("title").alias("title"), "yearReleaseDate", "releaseDate", "durationTime")    


             
#Colocamos los 2 dataFrame en una vista temporal localmente
movie_day1_df.createOrReplaceTempView("movie_day1")
movie_day2_df.createOrReplaceTempView("movie_day2")



In [0]:
%sql
--tabla en donde se almacenan los datos
Create table if not exists movie_demo.movies_merge(
        movieId INT,
        title STRING,
        yearReleaseDate INT,
        releaseDate DATE,
        durationTime INT,
        createdDate DATE,
        updatedDate DATE
)


In [0]:
%sql
--Merge SQL INSERTANDO LOS DATOS DE LA PRIMERA VISTA O DIA 1
MERGE INTO movie_demo.movies_merge AS A
USING movie_day1 AS B
ON A.movieId = B.movieId
WHEN MATCHED THEN
    UPDATE SET 
        A.title = B.title,
        A.yearReleaseDate = B.yearReleaseDate,
        A.releaseDate = B.releaseDate,
        A.durationTime = B.durationTime,
        A.updatedDate = current_timestamp
WHEN NOT MATCHED
    THEN INSERT(A.movieId, A.title, A.yearReleaseDate, A.releaseDate, A.durationTime, A.createdDate)
    VALUES(B.movieId, B.title, B.yearReleaseDate, B.releaseDate, B.durationTime, current_timestamp)
    

In [0]:
%sql
--Merge SQL INSERTANDO LOS DATOS DE LA SEGUNDA VISTA O DIA 2
MERGE INTO movie_demo.movies_merge AS A
USING movie_day2 AS B
ON A.movieId = B.movieId
WHEN MATCHED THEN
    UPDATE SET 
        A.title = B.title,
        A.yearReleaseDate = B.yearReleaseDate,
        A.releaseDate = B.releaseDate,
        A.durationTime = B.durationTime,
        A.updatedDate = current_timestamp
WHEN NOT MATCHED
    THEN INSERT(A.movieId, A.title, A.yearReleaseDate, A.releaseDate, A.durationTime, A.createdDate)
    VALUES(B.movieId, B.title, B.yearReleaseDate, B.releaseDate, B.durationTime, current_timestamp)

In [0]:
###Merge usando Python
#Insertamos el dia 3
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "movie_demo.movies_merge")

deltaTable.alias("A").merge(  ##tareget
    movie_day3_df.alias("B"), ##source
    "A.movieId = B.movieId"
    ).whenMatchedUpdate(
                        set = {
                                "A.title": "B.title",
                                "A.yearReleaseDate": "B.yearReleaseDate",
                                "A.releaseDate": "B.releaseDate",
                                "A.durationTime": "B.durationTime",
                                "A.updatedDate": "current_timestamp()"
                            }
                        )\
     .whenNotMatchedInsert(
                            values = {
                                        "A.movieId": "B.movieId",
                                        "A.title": "B.title",
                                        "A.yearReleaseDate": "B.yearReleaseDate",
                                        "A.releaseDate": "B.releaseDate",
                                        "A.durationTime": "B.durationTime",
                                        "A.createdDate": "current_timestamp()"
                                     }
                           )\
    .execute()


In [0]:
%sql
SELECT * FROM movie_demo.movies_merge

##HISTORY, TIME TRAVEL y VACUUM

In [0]:
%sql
--Podemos ver la Historia de la tabla y todas sus versiones a largo de su vida
DESC HISTORY movie_demo.movies_merge;

In [0]:
%sql
SELECT * FROM movie_demo.movies_merge timestamp as of '2026-08-25T22:50:53.000+00:00';

In [0]:
#Cargamos los datos de las versiones en un DataFrame
df = spark.read.format("delta").option("timestampAsOf", "2026-08-25T22:50:53.000+00:00").table("movie_demo.movies_merge")
df.display()

In [0]:
%sql
--VACUUM; para eliminar las versiones antiguas de la tabla
VACUUM movie_demo.movies_merge;
--No se llega a eliminar, porque internamente se mantene la historia
SELECT * FROM movie_demo.movies_merge timestamp as of '2026-08-25T22:50:53.000+00:00';



In [0]:
%sql
--podemos forzar la eliminacion
set spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM movie_demo.movies_merge RETAIN 0 HOURS;


In [0]:
%sql
SELECT * FROM movie_demo.movies_merge;

In [0]:
%sql
desc history movie_demo.movies_merge

In [0]:
%sql
--eliminamos 4 registros
delete from movie_demo.movies_merge where yearReleaseDate = 2004

In [0]:
%sql
--revisamos el versionado para ver cual es la ultima version antes de la eliminacion
desc history movie_demo.movies_merge

In [0]:
%sql
--En esta version se cuentran los 4 registros que se eliminaron con el delete
select * from movie_demo.movies_merge version as of 8 

In [0]:
%sql
--Merge SQL INSERTANDO LOS DATOS de lA VERSION 8 EN LA TABLA
--DEBE INSERTAR 4 REGISTROS, LOS CUALES SE BORRARON CON EL DELETE
MERGE INTO movie_demo.movies_merge AS A
USING movie_demo.movies_merge version as of 8  AS B
ON A.movieId = B.movieId
WHEN NOT MATCHED THEN
    INSERT * 

##Transacciones Log en Delta Lake

In [0]:
%sql
--tabla en donde se almacenan los datos
Create table if not exists movie_demo.movies_log(
        movieId INT,
        title STRING,
        yearReleaseDate INT,
        releaseDate DATE,
        durationTime INT,
        createdDate DATE,
        updatedDate DATE
)
using delta


In [0]:
%sql
desc history movie_demo.movies_log

In [0]:
%sql
describe extended movie_demo.movies_log

In [0]:
%sql
--Insertamos un registros en la tabla
insert into movie_demo.movies_log
select * from movie_demo.movies_merge 
where movieId = 125537


In [0]:
%sql
desc history movie_demo.movies_log

In [0]:
%sql
--Insertamos un registros en la tabla
insert into movie_demo.movies_log
select * from movie_demo.movies_merge 
where movieId = 133575

In [0]:
%sql
desc history movie_demo.movies_log

In [0]:
%sql
--Insertamos un registros en la tabla
delete from movie_demo.movies_log
where movieId = 125537;

In [0]:
%sql
desc history movie_demo.movies_log

In [0]:
#Insertamos la siguiente lista a la tala movie log
list = [118452, 124606, 125052, 125123, 125263, 125537, 126141, 133575, 142132, 146269, 157185]
for movieId in list:
    spark.sql(f"""INSERT INTO movie_demo.movies_log
        SELECT * FROM movie_demo.movies_merge
        WHERE movieId = {movieId}""")

In [0]:
%sql
INSERT INTO movie_demo.movies_log
SELECT * FROM movie_demo.movies_merge

In [0]:
%sql
desc history movie_demo.movies_log

##Convertir una tabla externa o directorio de formato parquet a formato delta

In [0]:
%sql
--creamos una tabla externa
Create table if not exists movie_demo.movies_convert_to_delta(
        movieId INT,
        title STRING,
        yearReleaseDate INT,
        releaseDate DATE,
        durationTime INT,
        createdDate DATE,
        updatedDate DATE
)
using parquet
LOCATION 'abfss://demo@lsdata01.dfs.core.windows.net/movies_convert_to_delta'


In [0]:
%sql
insert into movie_demo.movies_convert_to_delta
select * from movie_demo.movies_merge; 

In [0]:
%sql
--convertimos a formato delta
CONVERT TO DELTA movie_demo.movies_convert_to_delta;

In [0]:
df = spark.table('movie_demo.movies_convert_to_delta')
df.write.format("parquet").save("abfss://demo@lsdata01.dfs.core.windows.net/movies_convert_to_delta_new")

In [0]:
%sql
convert to delta parquet.`abfss://demo@lsdata01.dfs.core.windows.net/movies_convert_to_delta_new`